# Imports

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import re
import os
from collections import defaultdict
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
import numpy as np

# Connecting to database

In [2]:
DB_PATH = r"C:\Users\asule\Desktop\Task_DS\drilling_reports.db"
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
print("Connected to database!")

Connected to database!


# Explore wellbores and operations

In [3]:
# Checking all wellbores in database
print("=== WELLBORES IN DATABASE ===")
cur.execute("SELECT DISTINCT wellbore FROM reports ORDER BY wellbore")
for w in cur.fetchall():
    print(f"  {w[0]}")

# Checking operations per well of interest
wells_of_interest = ['15/9-F-10', '15/9-F-11', '15/9-F-12', '15/9-F-13']

print("\n=== OPERATIONS COUNT PER NDS WELL ===")
for well in wells_of_interest:
    cur.execute("""
        SELECT COUNT(*) FROM operations o
        JOIN reports r ON o.report_id = r.id
        WHERE r.wellbore = ?
    """, (well,))
    count = cur.fetchone()[0]

    cur.execute("""
        SELECT COUNT(DISTINCT r.id) FROM reports r
        WHERE r.wellbore = ?
    """, (well,))
    pdf_count = cur.fetchone()[0]

    print(f"  {well:<15} = {count:>4} operations across {pdf_count} PDFs")

=== WELLBORES IN DATABASE ===
  15/9-19 A
  15/9-19 B
  15/9-19 BT2
  15/9-19 S
  15/9-19 ST2
  15/9-F-10
  15/9-F-11
  15/9-F-11 A
  15/9-F-11 B
  15/9-F-11 T2
  15/9-F-12
  15/9-F-14
  15/9-F-15
  15/9-F-15 A

=== OPERATIONS COUNT PER NDS WELL ===
  15/9-F-10       = 1054 operations across 71 PDFs
  15/9-F-11       =  200 operations across 17 PDFs
  15/9-F-12       = 2043 operations across 165 PDFs
  15/9-F-13       =    0 operations across 0 PDFs


# Loading NDS events

In [4]:
nds_path = r"C:\Users\asule\Desktop\Task_DS\nds_events.xlsx"
nds_df = pd.read_excel(nds_path)
print("=== NDS EVENTS ===")
print(nds_df.to_string())

=== NDS EVENTS ===
        Well                                                                                                                                                                             Event
0  15/9-F-10                                                                                                    inclination angle was higher than expected. Reduced inclination to to 0.16 deg
1  15/9-F-11                                                                                                                  tight hole event was ecountered while drilling interval at 958 m
2  15/9-F-12                                                                                                          Excessive clay amount ccumulation is observed in BHA while POOH 26"" BHA
3  15/9-F-13  while RIH with 20" casing there was a differential stuck. The mud was replaced to seawater and 300 m3 of seawater was pumped, Set down weight was increased and string came free


# Loading operations from database for matching

In [5]:
query = """
    SELECT
        r.source_file,
        r.wellbore,
        r.period,
        o.id as op_id,
        o.start_time,
        o.end_time,
        o.end_depth_mmd,
        o.main_sub_activity,
        o.remark
    FROM operations o
    JOIN reports r ON o.report_id = r.id
    WHERE r.wellbore IN (
        '15/9-F-10',
        '15/9-F-11', '15/9-F-11 A', '15/9-F-11 B', '15/9-F-11 T2',
        '15/9-F-12'
    )
    AND o.remark IS NOT NULL
"""

ops_df = pd.read_sql_query(query, conn)

# Map all F-11 variants to '15/9-F-11' for NDS matching
ops_df['nds_well'] = ops_df['wellbore'].apply(
    lambda x: '15/9-F-11' if x.startswith('15/9-F-11') else x
)

print("=== OPERATIONS LOADED ===")
print(f"Total: {len(ops_df)} rows")

=== OPERATIONS LOADED ===
Total: 4890 rows


# Preprocessing

In [6]:
def preprocess(text):
    if not text:
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Now apply preprocess to ops_df
ops_df['text'] = (
    ops_df['main_sub_activity'].fillna('') + ' ' +
    ops_df['remark'].fillna('')
).apply(preprocess)

# Preprocess NDS events
nds_df['text'] = nds_df['Event'].apply(preprocess)

print("=== UPDATED OPERATIONS COUNT ===")
for well in ['15/9-F-10', '15/9-F-11', '15/9-F-12']:
    count = len(ops_df[ops_df['nds_well'] == well])
    print(f"  {well}: {count} operations")

print("\nNDS event texts:")
for _, row in nds_df.iterrows():
    print(f"  [{row['Well']}] {row['text'][:100]}")

=== UPDATED OPERATIONS COUNT ===
  15/9-F-10: 1002 operations
  15/9-F-11: 1910 operations
  15/9-F-12: 1978 operations

NDS event texts:
  [15/9-F-10] inclination angle was higher than expected reduced inclination to to 0 16 deg
  [15/9-F-11] tight hole event was ecountered while drilling interval at 958 m
  [15/9-F-12] excessive clay amount ccumulation is observed in bha while pooh 26 bha
  [15/9-F-13] while rih with 20 casing there was a differential stuck the mud was replaced to seawater and 300 m3 


- TF-IDF + Cosine Similarity — classic, fast, word frequency based
- BM25 — improved TF-IDF, better for short queries
- Semantic similarity (sentence-transformers) — deep learning, understands meaning

In [7]:
!pip install rank-bm25 sentence-transformers -q

# TF-IDF matching

In [8]:
def match_tfidf(nds_text, well, ops_df, top_n=1):
    """
    TF-IDF + Cosine Similarity matching.
    WHY: Fast, interpretable, works well for keyword overlap.
    Builds a term-frequency matrix and finds most similar document.
    """
    # Filter operations for this well only
    well_ops = ops_df[ops_df['wellbore'] == well].copy()
    if len(well_ops) == 0:
        return None

    corpus = well_ops['text'].tolist()

    # Fit TF-IDF on corpus + query together
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    all_texts = corpus + [nds_text]
    tfidf_matrix = vectorizer.fit_transform(all_texts)

    # Query is last row, corpus is everything before
    query_vec = tfidf_matrix[-1]
    corpus_matrix = tfidf_matrix[:-1]

    scores = cosine_similarity(query_vec, corpus_matrix).flatten()
    best_idx = np.argmax(scores)
    best_score = scores[best_idx]
    best_op = well_ops.iloc[best_idx]

    return {
        'method': 'TF-IDF',
        'well': well,
        'score': round(float(best_score), 4),
        'source_file': best_op['source_file'],
        'period': best_op['period'],
        'activity': best_op['main_sub_activity'],
        'matched_remark': best_op['remark'],
    }

print("TF-IDF function ready!")

TF-IDF function ready!


# BM25 matching

In [9]:
def match_bm25(nds_text, well, ops_df, top_n=1):
    """
    BM25 (Best Match 25) matching.
    WHY: Improved version of TF-IDF. Better handles term saturation
    and document length normalization. Standard in information retrieval.
    """
    well_ops = ops_df[ops_df['wellbore'] == well].copy()
    if len(well_ops) == 0:
        return None

    # BM25 needs tokenized corpus
    tokenized_corpus = [text.split() for text in well_ops['text'].tolist()]
    tokenized_query = nds_text.split()

    bm25 = BM25Okapi(tokenized_corpus)
    scores = bm25.get_scores(tokenized_query)

    best_idx = np.argmax(scores)
    best_score = scores[best_idx]
    best_op = well_ops.iloc[best_idx]

    return {
        'method': 'BM25',
        'well': well,
        'score': round(float(best_score), 4),
        'source_file': best_op['source_file'],
        'period': best_op['period'],
        'activity': best_op['main_sub_activity'],
        'matched_remark': best_op['remark'],
    }

print("BM25 function ready!")

BM25 function ready!


# Semantic matching

In [10]:
from sentence_transformers import SentenceTransformer, util

# Load pre-trained model — understands meaning not just keywords
# WHY: Can match "tight hole" with "stuck pipe" even without shared words
print("Loading semantic model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded!")

def match_semantic(nds_text, well, ops_df):
    """
    Semantic similarity using sentence transformers.
    WHY: Captures meaning beyond keywords. 'inclination' and 'angle'
    are recognized as related even without exact word match.
    Uses cosine similarity on dense vector embeddings.
    """
    well_ops = ops_df[ops_df['wellbore'] == well].copy()
    if len(well_ops) == 0:
        return None

    corpus = well_ops['text'].tolist()

    # Encode all operation texts + query
    print(f"  Encoding {len(corpus)} operations for {well}...")
    corpus_embeddings = model.encode(corpus, batch_size=64, show_progress_bar=False)
    query_embedding = model.encode([nds_text])

    scores = util.cos_sim(query_embedding, corpus_embeddings).numpy().flatten()

    best_idx = np.argmax(scores)
    best_score = scores[best_idx]
    best_op = well_ops.iloc[best_idx]

    return {
        'method': 'Semantic',
        'well': well,
        'score': round(float(best_score), 4),
        'source_file': best_op['source_file'],
        'period': best_op['period'],
        'activity': best_op['main_sub_activity'],
        'matched_remark': best_op['remark'],
    }

print("Semantic function ready!")

C:\Users\asule\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading semantic model...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 2135.28it/s]


Model loaded!
Semantic function ready!


# Running all algorithms and benchmark

In [11]:
results = []

print("=== RUNNING NDS MATCHING ===\n")

for _, nds_row in nds_df.iterrows():
    well = nds_row['Well']
    event = nds_row['Event']
    nds_text = nds_row['text']

    print(f"NDS Event [{well}]: {event[:80]}...")

    # Check if well exists in DB
    if well not in ops_df['wellbore'].unique():
        print(f"  !!! Well {well} not found in database!\n")
        results.append({
            'nds_well': well,
            'nds_event': event,
            'method': 'ALL',
            'score': None,
            'source_file': 'WELL NOT IN DATABASE',
            'period': None,
            'activity': None,
            'matched_remark': None,
        })
        continue

    # Run all 3 methods
    r_tfidf    = match_tfidf(nds_text, well, ops_df)
    r_bm25     = match_bm25(nds_text, well, ops_df)
    r_semantic = match_semantic(nds_text, well, ops_df)

    for r in [r_tfidf, r_bm25, r_semantic]:
        if r:
            r['nds_well'] = well
            r['nds_event'] = event
            results.append(r)

    print(f"  TF-IDF   score={r_tfidf['score']:.4f}  file={r_tfidf['source_file']}")
    print(f"  BM25     score={r_bm25['score']:.4f}  file={r_bm25['source_file']}")
    print(f"  Semantic score={r_semantic['score']:.4f}  file={r_semantic['source_file']}")
    print()

results_df = pd.DataFrame(results)
print("=== DONE ===")
print(f"Total results: {len(results_df)}")

=== RUNNING NDS MATCHING ===

NDS Event [15/9-F-10]: inclination angle was higher than expected. Reduced inclination to to 0.16 deg...
  Encoding 1002 operations for 15/9-F-10...
  TF-IDF   score=0.1875  file=15_9_F_10_2009_04_12.pdf
  BM25     score=26.4108  file=15_9_F_10_2009_04_12.pdf
  Semantic score=0.3462  file=15_9_F_10_2009_04_11.pdf

NDS Event [15/9-F-11]: tight hole event was ecountered while drilling interval at 958 m...
  Encoding 187 operations for 15/9-F-11...
  TF-IDF   score=0.0627  file=15_9_F_11_2013_03_17.pdf
  BM25     score=6.9443  file=15_9_F_11_2013_03_13.pdf
  Semantic score=0.6350  file=15_9_F_11_2013_03_08.pdf

NDS Event [15/9-F-12]: Excessive clay amount ccumulation is observed in BHA while POOH 26"" BHA...
  Encoding 1978 operations for 15/9-F-12...
  TF-IDF   score=0.1641  file=15_9_F_12_2007_07_06.pdf
  BM25     score=16.6175  file=15_9_F_12_2007_07_12.pdf
  Semantic score=0.4658  file=15_9_F_12_2007_08_18.pdf

NDS Event [15/9-F-13]: while RIH with 20" ca

# Results

In [12]:
print("=== FINAL MATCHING RESULTS ===\n")

for _, nds_row in nds_df.iterrows():
    well = nds_row['Well']
    event = nds_row['Event']
    print(f"{'='*70}")
    print(f"NDS Event [{well}]:")
    print(f"  {event}")
    print()

    well_results = results_df[results_df['nds_well'] == well]

    for _, r in well_results.iterrows():
        if r['source_file'] == 'WELL NOT IN DATABASE':
            print(f"!!! Well not found in database")
            break
        print(f"  [{r['method']}] Score: {r['score']}")
        print(f"  File:     {r['source_file']}")
        print(f"  Period:   {r['period']}")
        print(f"  Activity: {r['activity']}")
        print(f"  Matched:  {str(r['matched_remark'])[:200]}")
        print()

=== FINAL MATCHING RESULTS ===

NDS Event [15/9-F-10]:
  inclination angle was higher than expected. Reduced inclination to to 0.16 deg

  [TF-IDF] Score: 0.1875
  File:     15_9_F_10_2009_04_12.pdf
  Period:   2009-04-11 00:00 - 2009-04-12 00:00
  Activity: drilling --drill
  Matched:  Reamed interval 222-246 m MD in order to reduce inclination. Parameters : Flow 3500 lpm / SPP 95 bar / 140 RPM / Torque 2-4 kNm. Pumped 2 x 10 m3 hivis pills. Reduced iclination from 0,53 deg to 0,16 

  [BM25] Score: 26.4108
  File:     15_9_F_10_2009_04_12.pdf
  Period:   2009-04-11 00:00 - 2009-04-12 00:00
  Activity: drilling --drill
  Matched:  Reamed interval 222-246 m MD in order to reduce inclination. Parameters : Flow 3500 lpm / SPP 95 bar / 140 RPM / Torque 2-4 kNm. Pumped 2 x 10 m3 hivis pills. Reduced iclination from 0,53 deg to 0,16 

  [Semantic] Score: 0.3462
  File:     15_9_F_10_2009_04_11.pdf
  Period:   2009-04-10 00:00 - 2009-04-11 00:00
  Activity: drilling -- casing
  Matched:  Sla

In [13]:
print("=== ALGORITHM BENCHMARK COMPARISON ===\n")

for _, nds_row in nds_df.iterrows():
    well = nds_row['Well']
    event = nds_row['Event']

    print(f"Well: {well}")
    print(f"Event: {event[:80]}...")

    well_results = results_df[results_df['nds_well'] == well]

    for _, r in well_results.iterrows():
        if r['source_file'] == 'WELL NOT IN DATABASE':
            print(f"  !!! Well not found in database")
            break
        print(f"  {r['method']:<10} score={r['score']:<8} file={r['source_file']}")
    print()

print("\n=== ALGORITHM SUMMARY ===")
print("TF-IDF:   Fast, keyword-based, scores between 0-1")
print("BM25:     Fast, better than TF-IDF for short queries, scores can exceed 1")
print("Semantic: Slow, understands meaning, scores between 0-1")
print("\nAgreement between methods:")
for _, nds_row in nds_df.iterrows():
    well = nds_row['Well']
    well_results = results_df[results_df['nds_well'] == well]
    files = well_results['source_file'].tolist()
    if 'WELL NOT IN DATABASE' in files:
        print(f"  {well}: well not in database")
        continue
    unique_files = set(files)
    agreement = "ALL AGREE" if len(unique_files) == 1 else f"{len(unique_files)} different files"
    print(f"  {well}: {agreement} — {files}")

=== ALGORITHM BENCHMARK COMPARISON ===

Well: 15/9-F-10
Event: inclination angle was higher than expected. Reduced inclination to to 0.16 deg...
  TF-IDF     score=0.1875   file=15_9_F_10_2009_04_12.pdf
  BM25       score=26.4108  file=15_9_F_10_2009_04_12.pdf
  Semantic   score=0.3462   file=15_9_F_10_2009_04_11.pdf

Well: 15/9-F-11
Event: tight hole event was ecountered while drilling interval at 958 m...
  TF-IDF     score=0.0627   file=15_9_F_11_2013_03_17.pdf
  BM25       score=6.9443   file=15_9_F_11_2013_03_13.pdf
  Semantic   score=0.635    file=15_9_F_11_2013_03_08.pdf

Well: 15/9-F-12
Event: Excessive clay amount ccumulation is observed in BHA while POOH 26"" BHA...
  TF-IDF     score=0.1641   file=15_9_F_12_2007_07_06.pdf
  BM25       score=16.6175  file=15_9_F_12_2007_07_12.pdf
  Semantic   score=0.4658   file=15_9_F_12_2007_08_18.pdf

Well: 15/9-F-13
Event: while RIH with 20" casing there was a differential stuck. The mud was replaced t...
  !!! Well not found in database


# Additional Methods to make results better

## Query expansion

In [14]:
# Domain-specific drilling terminology expansion
DRILLING_SYNONYMS = {
    'tight hole':     ['overpull', 'drag', 'stuck', 'resistance', 'high torque', 'reaming', 'tight'],
    'stuck':          ['differential stuck', 'mechanically stuck', 'pack off', 'unable to move'],
    'inclination':    ['angle', 'deviation', 'azimuth', 'survey', 'directional'],
    'clay':           ['shale', 'formation', 'cuttings', 'buildup', 'accumulation', 'balling'],
    'bha':            ['bottom hole assembly', 'drill string', 'stabilizer', 'bit'],
    'pooh':           ['pull out of hole', 'tripping out', 'trip out'],
    'rih':            ['run in hole', 'tripping in', 'trip in'],
    'casing':         ['liner', 'string', 'pipe'],
    'mud':            ['fluid', 'drilling fluid', 'obm', 'wbm', 'seawater'],
    'differential':   ['pressure differential', 'stuck', 'overbalance'],
    'drilling':       ['drill', 'rotate', 'wob', 'rop'],
    'interval':       ['section', 'hole section', 'depth'],
}

def expand_query(text):
    """
    Expand query with domain synonyms.
    WHY: NDS events are written in high-level language,
    operation remarks use field jargon. Expansion bridges the gap.
    """
    expanded = text.lower()
    additions = []
    for term, synonyms in DRILLING_SYNONYMS.items():
        if term in expanded:
            additions.extend(synonyms)
    if additions:
        expanded = expanded + ' ' + ' '.join(additions)
    return expanded

test = "tight hole event was encountered while drilling interval at 958 m"
print("Original:", test)
print("Expanded:", expand_query(test))

Original: tight hole event was encountered while drilling interval at 958 m
Expanded: tight hole event was encountered while drilling interval at 958 m overpull drag stuck resistance high torque reaming tight drill rotate wob rop section hole section depth


## Depth extraction

In [15]:
def extract_depth(text):
    """
    Extract depth values from text.
    WHY: Depth is a critical matching signal in drilling operations.
    If NDS event mentions 958m, we prioritize operations near that depth.
    """
    if not text:
        return []
    pattern = r'(\d+(?:\.\d+)?)\s*m(?:MD|TVD|D)?\b'
    depths = re.findall(pattern, text, re.IGNORECASE)
    return [float(d) for d in depths]

def depth_similarity(depths1, depths2, tolerance=50):
    """
    Check if two sets of depths are similar within 50m tolerance.
    Returns 1.0 if any depth pair is within tolerance, 0.0 otherwise.
    """
    if not depths1 or not depths2:
        return 0.0
    for d1 in depths1:
        for d2 in depths2:
            if abs(d1 - d2) <= tolerance:
                return 1.0
    return 0.0

event = "tight hole event was encountered while drilling interval at 958 m"
remark = "Reamed from 950 m to 970 m, encountered tight hole"
print(f"Event depths:     {extract_depth(event)}")
print(f"Remark depths:    {extract_depth(remark)}")
print(f"Depth similarity: {depth_similarity(extract_depth(event), extract_depth(remark))}")

Event depths:     [958.0]
Remark depths:    [950.0, 970.0]
Depth similarity: 1.0


# Load summaries

In [16]:
query_summaries = """
    SELECT
        r.source_file,
        r.wellbore,
        r.period,
        r.summary_24h as text,
        'summary_24h' as text_type
    FROM reports r
    WHERE r.wellbore IN (
        '15/9-F-10',
        '15/9-F-11', '15/9-F-11 A', '15/9-F-11 B', '15/9-F-11 T2',
        '15/9-F-12'
    )
    AND r.summary_24h IS NOT NULL

    UNION ALL

    SELECT
        r.source_file,
        r.wellbore,
        r.period,
        r.summary_planned as text,
        'summary_planned' as text_type
    FROM reports r
    WHERE r.wellbore IN (
        '15/9-F-10',
        '15/9-F-11', '15/9-F-11 A', '15/9-F-11 B', '15/9-F-11 T2',
        '15/9-F-12'
    )
    AND r.summary_planned IS NOT NULL
"""

summaries_df = pd.read_sql_query(query_summaries, conn)
summaries_df['nds_well'] = summaries_df['wellbore'].apply(
    lambda x: '15/9-F-11' if x.startswith('15/9-F-11') else x
)
summaries_df['text_clean'] = summaries_df['text'].apply(preprocess)

print(f"Loaded {len(summaries_df)} summary sections")
print(summaries_df['nds_well'].value_counts())

Loaded 820 summary sections
nds_well
15/9-F-11    348
15/9-F-12    330
15/9-F-10    142
Name: count, dtype: int64


## Hybrid function

In [17]:
def match_hybrid(nds_event, well, ops_df, summaries_df, model,
                 w_tfidf=0.25, w_bm25=0.35, w_semantic=0.40):
    """
    Hybrid matching: TF-IDF + BM25 + Semantic + Depth boost.
    Searches both operations AND summaries for maximum coverage.
    Uses nds_well column to handle well variants (e.g. F-11 A, B, T2).
    """
    expanded_query = expand_query(preprocess(nds_event))
    nds_depths = extract_depth(nds_event)

    # Filter using nds_well (handles all variants)
    well_ops = ops_df[ops_df['nds_well'] == well].copy()
    well_ops['text_type'] = 'operation'
    well_ops['text_clean'] = well_ops['text']

    well_sums = summaries_df[summaries_df['nds_well'] == well].copy()

    search_corpus = pd.concat([
        well_ops[['source_file', 'wellbore', 'period', 'text_clean', 'text_type', 'remark']],
        well_sums[['source_file', 'wellbore', 'period', 'text_clean', 'text_type']].assign(remark=well_sums['text'])
    ], ignore_index=True)

    if len(search_corpus) == 0:
        return None

    corpus = search_corpus['text_clean'].tolist()

    # TF-IDF
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    tfidf_matrix = vectorizer.fit_transform(corpus + [expanded_query])
    tfidf_scores = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1]).flatten()
    if tfidf_scores.max() > 0:
        tfidf_scores = tfidf_scores / tfidf_scores.max()

    # BM25
    tokenized_corpus = [t.split() for t in corpus]
    bm25 = BM25Okapi(tokenized_corpus)
    bm25_scores = bm25.get_scores(expanded_query.split())
    if bm25_scores.max() > 0:
        bm25_scores = bm25_scores / bm25_scores.max()

    # Semantic
    corpus_embeddings = model.encode(corpus, batch_size=64, show_progress_bar=False)
    query_embedding = model.encode([expanded_query])
    semantic_scores = util.cos_sim(query_embedding, corpus_embeddings).numpy().flatten()

    # Depth boost
    depth_boost = np.array([
        depth_similarity(nds_depths, extract_depth(str(text)))
        for text in search_corpus['remark'].fillna('')
    ])

    # Hybrid score
    hybrid_scores = (
        w_tfidf    * tfidf_scores +
        w_bm25     * bm25_scores +
        w_semantic * semantic_scores +
        0.15       * depth_boost
    )

    best_idx = np.argmax(hybrid_scores)
    best_row = search_corpus.iloc[best_idx]

    return {
        'nds_event':      nds_event,
        'well':           well,
        'method':         'Hybrid',
        'hybrid_score':   round(float(hybrid_scores[best_idx]), 4),
        'tfidf_score':    round(float(tfidf_scores[best_idx]), 4),
        'bm25_score':     round(float(bm25_scores[best_idx]), 4),
        'semantic_score': round(float(semantic_scores[best_idx]), 4),
        'depth_boost':    round(float(depth_boost[best_idx]), 4),
        'text_type':      best_row['text_type'],
        'source_file':    best_row['source_file'],
        'period':         best_row['period'],
        'matched_text':   best_row['remark'],
        'expanded_query': expanded_query,
        'nds_depths':     nds_depths,
        'actual_wellbore': best_row['wellbore'],
    }

print("Updated hybrid function ready!")

Updated hybrid function ready!


# Running Hybrid

In [18]:
print("=== HYBRID NDS MATCHING ===\n")

hybrid_results = []

for _, nds_row in nds_df.iterrows():
    well = nds_row['Well']
    event = nds_row['Event']

    print(f"[{well}] {event[:70]}...")

    if well not in ops_df['wellbore'].unique():
        print(f" !!! Well not found in database\n")
        hybrid_results.append({
            'nds_event': event, 'well': well, 'method': 'Hybrid',
            'hybrid_score': None, 'tfidf_score': None, 'bm25_score': None,
            'semantic_score': None, 'depth_boost': None,
            'source_file': 'WELL NOT IN DATABASE', 'period': None,
            'text_type': None, 'matched_text': None,
            'expanded_query': None, 'nds_depths': None,
        })
        continue

    result = match_hybrid(event, well, ops_df, summaries_df, model)
    hybrid_results.append(result)

    print(f"  Hybrid score:  {result['hybrid_score']}")
    print(f"  TF-IDF:        {result['tfidf_score']}")
    print(f"  BM25:          {result['bm25_score']}")
    print(f"  Semantic:      {result['semantic_score']}")
    print(f"  Depth boost:   {result['depth_boost']}")
    print(f"  Match type:    {result['text_type']}")
    print(f"  File:          {result['source_file']}")
    print(f"  Matched text:  {str(result['matched_text'])[:200]}")
    print()

hybrid_df = pd.DataFrame(hybrid_results)
print("=== DONE ===")

=== HYBRID NDS MATCHING ===

[15/9-F-10] inclination angle was higher than expected. Reduced inclination to to ...
  Hybrid score:  0.7404
  TF-IDF:        0.9696
  BM25:          1.0
  Semantic:      0.3701
  Depth boost:   0.0
  Match type:    operation
  File:          15_9_F_10_2009_04_08.pdf
  Matched text:  Drilled 36" hole from 165 m to 207 m (17 1/2" bit depth). 90-100 RPM, 5 ton WOB, 4000-4450 lpm, 3,7 kNm, ROP 5-20 m/hr. Pumped HIVIS pills as per program.Took survey at 170 m: Inclination 0,27 deg.Too

[15/9-F-11] tight hole event was ecountered while drilling interval at 958 m...
  Hybrid score:  0.7958
  TF-IDF:        1.0
  BM25:          0.8583
  Semantic:      0.6135
  Depth boost:   0.0
  Match type:    operation
  File:          15_9_F_11_T2_2013_04_02.pdf
  Matched text:  Pulled out of hole with 26" bottom hole assembly from 1365m MD to 460m MD. Observed tight hole at 460m MD. Attempted to get past restriction with maximum 30MToverpull - no go.

[15/9-F-12] Excessive c

# Final benchmark all methods

In [19]:
print("=== FINAL BENCHMARK: ALL METHODS ===\n")

for _, nds_row in nds_df.iterrows():
    well = nds_row['Well']
    event = nds_row['Event']

    print(f"{'='*65}")
    print(f"Well: {well}")
    print(f"NDS Event: {event}")
    print()

    if well not in ops_df['wellbore'].unique():
        print(" !!! Well not found in database\n")
        continue

    well_ind = results_df[results_df['nds_well'] == well]
    print("  Individual methods:")
    for _, r in well_ind.iterrows():
        print(f"    {r['method']:<10} score={r['score']:<8} file={r['source_file']}")

    hyb = hybrid_df[hybrid_df['well'] == well]
    if len(hyb) > 0:
        h = hyb.iloc[0]
        print(f"\n  Hybrid method:")
        print(f"    Score={h['hybrid_score']}  file={h['source_file']}")
        print(f"    Match type: {h['text_type']}")
        print(f"    Matched: {str(h['matched_text'])[:200]}")
    print()

=== FINAL BENCHMARK: ALL METHODS ===

Well: 15/9-F-10
NDS Event: inclination angle was higher than expected. Reduced inclination to to 0.16 deg

  Individual methods:
    TF-IDF     score=0.1875   file=15_9_F_10_2009_04_12.pdf
    BM25       score=26.4108  file=15_9_F_10_2009_04_12.pdf
    Semantic   score=0.3462   file=15_9_F_10_2009_04_11.pdf

  Hybrid method:
    Score=0.7404  file=15_9_F_10_2009_04_08.pdf
    Match type: operation
    Matched: Drilled 36" hole from 165 m to 207 m (17 1/2" bit depth). 90-100 RPM, 5 ton WOB, 4000-4450 lpm, 3,7 kNm, ROP 5-20 m/hr. Pumped HIVIS pills as per program.Took survey at 170 m: Inclination 0,27 deg.Too

Well: 15/9-F-11
NDS Event: tight hole event was ecountered while drilling interval at 958 m

  Individual methods:
    TF-IDF     score=0.0627   file=15_9_F_11_2013_03_17.pdf
    BM25       score=6.9443   file=15_9_F_11_2013_03_13.pdf
    Semantic   score=0.635    file=15_9_F_11_2013_03_08.pdf

  Hybrid method:
    Score=0.7958  file=15_9_F_11_

# Saving to excel

In [20]:
# Saving full results to Excel
output_path = r"C:\Users\asule\Desktop\Task_DS\nds_matching_results.xlsx"

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:

    # Sheet 1: Individual methods
    export_individual = results_df[[
        'nds_well', 'nds_event', 'method', 'score',
        'source_file', 'period', 'activity', 'matched_remark'
    ]].copy()
    export_individual.columns = [
        'Well', 'NDS Event', 'Method', 'Score',
        'Matched PDF', 'Period', 'Activity', 'Matched Remark'
    ]
    export_individual.to_excel(writer, sheet_name='Individual Methods', index=False)

    # Sheet 2: Hybrid results
    export_hybrid = hybrid_df[[
        'well', 'nds_event', 'method', 'hybrid_score',
        'tfidf_score', 'bm25_score', 'semantic_score', 'depth_boost',
        'source_file', 'period', 'text_type', 'matched_text'
    ]].copy()
    export_hybrid.columns = [
        'Well', 'NDS Event', 'Method', 'Hybrid Score',
        'TF-IDF Score', 'BM25 Score', 'Semantic Score', 'Depth Boost',
        'Matched PDF', 'Period', 'Match Type', 'Matched Text'
    ]
    export_hybrid.to_excel(writer, sheet_name='Hybrid Method', index=False)

print(f"Saved to: {output_path}")
print("2 sheets: Individual Methods + Hybrid Method")

Saved to: C:\Users\asule\Desktop\Task_DS\nds_matching_results.xlsx
2 sheets: Individual Methods + Hybrid Method


# for checking

In [21]:
# Check what depths appear in F-11 operations
cur.execute("""
    SELECT o.end_depth_mmd, o.remark
    FROM operations o
    JOIN reports r ON o.report_id = r.id
    WHERE r.wellbore = '15/9-F-11'
    AND o.remark IS NOT NULL
    ORDER BY CAST(o.end_depth_mmd AS FLOAT) DESC
    LIMIT 10
""")
rows = cur.fetchall()
print("=== DEEPEST OPERATIONS IN 15/9-F-11 ===")
for depth, remark in rows:
    print(f"\n  Depth: {depth}m")
    print(f"  Remark: {remark[:150]}")

# Also check max depth in reports
cur.execute("""
    SELECT r.source_file, r.depth_mmd
    FROM reports r
    WHERE r.wellbore = '15/9-F-11'
    ORDER BY CAST(r.depth_mmd AS FLOAT) DESC
    LIMIT 5
""")
print("\n=== MAX DEPTHS IN F-11 REPORTS ===")
for f, d in cur.fetchall():
    print(f"  {f}: {d}m")

=== DEEPEST OPERATIONS IN 15/9-F-11 ===

  Depth: 347m
  Remark: RIH w/ cement stinger on 5 1/2" DP from 331 m, 1000 l/min, 4 bars and tagged TD @347 m with 2 tons. Picked off bottom and circulated 2 x bottoms up wi

  Depth: 345m
  Remark: Racked back stand and M/U cement stand. RIH to 345 m on cement stand.

  Depth: 345m
  Remark: M/U side entry sub and cement hose. Attempted to flush line from cement unit, observed pressure build up to 35 bar. Bled off pressure, disconnected ce

  Depth: 345m
  Remark: M/U cement stand and RIH to 2 m off bottom at 345 m w/1000 l/min, 5 bar. Connected and tested cement line, 20/100 bar for 5/10 min.

  Depth: 345m
  Remark: Mixed and pumped 78 m3 of 1.92 SG G neat cement slurry. Displaced cement with 2.1 m3 sea water with cement unit.

  Depth: 332m
  Remark: Orientated drilled 26" hole from bottom at 320 m to 332 m and recorded OnTrack survey data.Re-calculated projected well path.Inclination Survey (OnTra

  Depth: 331m
  Remark: RIH cement stinge